```sql
Goal:
- load the joined train / validation split
- build a simple baseline feature set
- train one baseline model
- evaluate on the validation split

```

In [1]:
import pandas as pd
import numpy as np

#### 1. Load Joined train validation split

In [2]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [3]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [4]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [5]:
print(f"Train fraud rate: {train_df['isFraud'].mean():.2%}")
print(f"Valid fraud rate: {valid_df['isFraud'].mean():.2%}")

Train fraud rate: 3.51%
Valid fraud rate: 3.44%


#### 2. Building Baseline Feature Set 

In [6]:
train_df.head(5)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [7]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [8]:
available_features = [c for c in base_features if c in train_df.columns]

In [9]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [10]:
new_features = ["log_txnamt", "has_identity"]

In [12]:
feature_cols=available_features+new_features

In [13]:
X_train=train_df[feature_cols]
X_valid=valid_df[feature_cols]

y_train=train_df[target_col]
y_valid=valid_df[target_col]

In [ ]:
#Counting Categorical and Numerical Columns